# Лекција 10 - AI агенти у продукцији

У овој лекцији ћете научити **производне шаблоне** за AI агенте коришћењем Microsoft Agent Framework-а (Python). Обрађујемо:

- **Посматрање** — додавање временског праћења и евиденције интеракција агента
- **Евалуација** — коришћење евалуатора агента за оцењивање квалитета одговора
- **Управљање трошковима** — стратегије за оптимизацију броја токена и избор модела

Сценарио је **туристички агент** који помаже корисницима у планирању путовања, уз надзор и евалуацију као додатак.


## Подешавање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Разматрања за производњу

Преношење AI агената из прототипова у производњу захтева пажљиво осмотрење три стуба:

1. **Посматрање** — Потребна вам је видљивост у оно што агент ради, колико му времена треба и које алате користи. Без праћења и логовања, отклањање грешака у производњи је скоро немогуће.

2. **Евалуација** — Аутоматизовани квалитетни прегледи обезбеђују да агентови одговори остају тачни, потпуни и корисни током времена. Евалуаторски агент може оцењивати одговоре према дефинисаним критеријумима.

3. **Управљање трошковима** — Коришћење токена директно утиче на трошкове. Стратегије као што су оптимизација упита, избор модела и кеширање помажу да се трошкови држе под контролом без жртвовања квалитета.


## Прављење посматрачког агента

Дефинишемо алате за путовање и обавијамо позив агента временским мерењем како бисмо могли да пратимо кашњење. У продукцији бисте интегрисали са OpenTelemetry или сличним системом за праћење.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Обрасци процене

Уобичајени производни образац је коришћење другог агента као **процењивача**. Процењивач оцењује одговор примарног агента према унапред дефинисаним критеријумима као што су потпуност, тачност и корисност.

Ово омогућава:
- Аутоматизоване контроле квалитета пре него што одговори стигну до корисника
- Откривање регресија када се промене упити или модели
- Континуирано праћење перформанси агента током времена


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегије управљања трошковима

Контрола трошкова је кључна за производне AI агенте. Ево кључних стратегија:

| Стратегија | Опис |
|---|---|
| **Оптимизација промпта** | Држите упутства система концизним. Уклоните сувишни контекст да бисте смањили улазне токене. |
    "| **Избор модела** | Користите мање, јефтиније моделе (нпр. GPT-5-mini) за једноставне задатке као што су класификација или извлачење, а веће моделе резервишите за сложене резоне. |\n",
| **Кешање** | Кеширајте резултате алата и честа питања да бисте избегли сувишне API позиве. |
| **Буџети токена** | Поставите лимите `max_tokens` да спречите неочекивано дуге одговоре. |
| **Паковање упита** | Групишите више корисничких упита у један API позив где је то могуће. |

У пракси, добро функционише слојевити приступ: усмерите једноставне захтеве ка брзом, јефтином моделу и само сложене упите прослеђујте способнијем моделу.


## Резиме

У овој лекцији сте научили како да:

1. **Додате видљивост** интеракцијама агента кроз мерење времена и евиденцију, постављајући темеље за праћење и мониторинг.
2. **Аутоматски процењујете одговоре агента** користећи агента за евалуацију који оцењује потпуност, тачност и корисност.
3. **Управљате трошковима** путем оптимизације захтева, избора модела, кеширања и буџета токена.

Ови производни обрасци помажу да ваши AI агенти буду поуздани, мерљиви и ефикасни у трошковима на великој скали.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
